In [0]:
from pyspark.sql import functions as f
bronze_df = spark.read.table("projects.ecommerce_project.raw_retail_online")
#To trim, standardise the casing on text fields 
standardised_df = bronze_df.withColumn("Description",f.trim(f.upper(f.col("Description"))))\
                            .withColumn("StockCode",f.trim(f.upper(f.col("StockCode"))))\
                            .withColumn("Country",f.trim(f.upper(f.col("Country"))))\
                            .withColumn("Invoice",f.trim(f.upper(f.col("Invoice"))))
#dropping Duplicated from the Dataframe                           
standardised_df = standardised_df.dropDuplicates()
print(f"Count :{standardised_df.count()}")

In [0]:
#creating a list of non product codes to filter out
Adjustment_codes = ["S","POST","PADS","DOT","D","B","BANK CHARGES","CRUK","C2","M","C3", "AMAZONFEE"]
transactional_df = standardised_df.filter(~f.col("StockCode").isin(Adjustment_codes))
adjustments_df = standardised_df.filter(f.col("StockCode").isin(Adjustment_codes))
#Sanity Check
transactional_df.filter(~f.col("StockCode").rlike("^[0-9]")).select("StockCode").distinct().show(50, truncate=False)

In [0]:
silver_df = transactional_df.withColumn("iscancelled", (f.col("Invoice")).startswith("C"))\
                            .withColumn("guestcheckout",(f.col("Customer_ID")).isNull())\
                            .withColumn("Customer_ID",(f.col("Customer_ID")).cast("string"))\
                            .withColumn("total_price", (f.col("Quantity")* f.col("Price")))\
                            .withColumnRenamed("Invoice","invoice_No")\
                            .withColumnRenamed("Country","country")\
                            .withColumnRenamed("Price","unti_price")\
                            .withColumnRenamed("StockCode","stock_code")\
                            .withColumnRenamed("Description","description")\
                            .withColumnRenamed("InvoiceDate","invoice_date")\
                            .withColumnRenamed("Quantity","quantity")\
                            .withColumnRenamed("Customer_ID","customer_id")
silver_df.printSchema()
display(silver_df.limit(10))
                        
                            

In [0]:
mismatches = silver_df.filter((f.col("quantity") < 0) & (~f.col("iscancelled")))
print(f"Number of mismatches : {mismatches.count()}")
mismatches.select("invoice_no", "stock_code", "quantity", "iscancelled").show(20)

In [0]:
silver_adjustments = adjustments_df \
    .withColumn("customer_id", f.col("Customer_ID").cast("string")) \
    .withColumn("total_price", f.col("Quantity") * f.col("Price")) \
    .withColumnRenamed("Invoice", "invoice_no") \
    .withColumnRenamed("StockCode", "adjustment_code") \
    .withColumnRenamed("Description", "description") \
    .withColumnRenamed("Quantity", "quantity") \
    .withColumnRenamed("InvoiceDate", "invoice_date") \
    .withColumnRenamed("Price", "unit_price") \
    .withColumnRenamed("Country", "country") \
    .drop("Customer_ID")

display(silver_adjustments.limit(10))

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("projects.ecommerce_project.silver_transactons")

silver_adjustments.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("projects.ecommerce_project.silver_adjustments")

In [0]:
print("Transactions:", spark.table("projects.ecommerce_project.silver_transactons").count())
print("Adjustments:", spark.table("projects.ecommerce_project.silver_adjustments").count())

# Should equal Bronze total excluding the duplicates which were removed previously
print("Bronze total:", bronze_df.count())